# Notebook 03 — Gold: Business Aggregation

**What we are doing:** reading the 75 clean Silver records and building each business query independently (sales count, quantity, revenue, average sale value — each per brand), then combining all four measures into one Gold table with one row per brand, and identifying the brand with the highest total revenue.

**Flow:** Read Silver records → Independent queries (sales count, quantity, revenue, avg sale value) → Combined query → Highest revenue brand → Verification

**Output — Gold business table:**

* `electronic_sales.gold.electronics_revenue_by_brand` — one row per brand: `total_sales_count`, `total_quantity`, `total_revenue`, `avg_sale_value`

**Cell 1 — Read silver records:** the 75 clean rows from `electronic_sales.silver.electronics_sales`.

In [0]:
from pyspark.sql.functions import *

silver_df = spark.read.table("electronic_sales.silver.electronics_sales")

print("Silver records into Gold =", silver_df.count())

**Cell 2 — Total sales count by brand (independent):** `count(sale_id)` per brand alone.

In [0]:
sales_count_df = silver_df.groupBy("brand") \
    .agg(count("sale_id").alias("total_sales_count"))

sales_count_df.orderBy(col("total_sales_count").desc()).show()

**Cell 3 — Total quantity by brand (independent):** `sum(quantity)` per brand alone.

In [0]:
quantity_df = silver_df.groupBy("brand") \
    .agg(sum("quantity").alias("total_quantity"))

quantity_df.orderBy(col("total_quantity").desc()).show()

**Cell 4 — Total revenue by brand (independent):** `round(sum(net_sales), 2)` per brand alone.

In [0]:
revenue_df = silver_df.groupBy("brand") \
    .agg(round(sum("net_sales"), 2).alias("total_revenue"))

revenue_df.orderBy(col("total_revenue").desc()).show()

**Cell 5 — Average sale value by brand (independent):** `round(avg(net_sales), 2)` per brand alone.

In [0]:
avg_value_df = silver_df.groupBy("brand") \
    .agg(round(avg("net_sales"), 2).alias("avg_sale_value"))

avg_value_df.orderBy(col("avg_sale_value").desc()).show()

**Cell 6 — Combined gold table:** all four measures in one query, one row per brand → `electronic_sales.gold.electronics_revenue_by_brand`.

In [0]:
gold_df = silver_df.groupBy("brand") \
    .agg(
        count("sale_id").alias("total_sales_count"),
        sum("quantity").alias("total_quantity"),
        round(sum("net_sales"), 2).alias("total_revenue"),
        round(avg("net_sales"), 2).alias("avg_sale_value")
    )

gold_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "electronic_sales.gold.electronics_revenue_by_brand"
    )

display(gold_df)

**Cell 7 — Highest revenue brand:** filters the brand with the maximum `total_revenue`.

In [0]:
max_revenue = gold_df.agg(max("total_revenue")).collect()[0][0]

highest_revenue_brand_df = gold_df.filter(
    col("total_revenue") == max_revenue
)

brand_row = highest_revenue_brand_df.collect()[0]

print("Highest revenue brand:", brand_row["brand"], "— total revenue:", max_revenue)

highest_revenue_brand_df.show(truncate=False)

**Cell 8 — Verify gold:** one row per brand; totals must reconcile with Silver.

In [0]:
gold_check = spark.table("electronic_sales.gold.electronics_revenue_by_brand")

print("Gold rows (one per brand) =", gold_check.count())
print("Revenue match:", gold_check.agg(round(sum("total_revenue"), 2)).collect()[0][0],
      "= silver total", silver_df.agg(round(sum("net_sales"), 2)).collect()[0][0])
print("Quantity match:", gold_check.agg(sum("total_quantity")).collect()[0][0],
      "= silver total", silver_df.agg(sum("quantity")).collect()[0][0])
print("Sales count match:", gold_check.agg(sum("total_sales_count")).collect()[0][0],
      "= silver total", silver_df.count())

display(gold_check)

**Gold layer complete:** brand metrics aggregated from 75 silver records, totals reconcile with Silver ✅